# 练习：基于进化策略的策略优化

本练习基于 `1_policy_optimization_introduction.ipynb`。目标不是机械复现示例，而是把“获得更优策略”这一模糊目标转化为可评价、可比较的实验问题。


## 1. 明确实验目标

首先定义什么叫“更优策略”。只看一次 rollout 的累计回报通常不够，因为环境初始状态和状态转移可能带有随机性。建议至少考虑：

- 多次 rollout 的平均累计回报；
- 最差若干次 rollout 的表现，用于观察鲁棒性；
- 学习过程中达到目标回报所需的环境交互次数；
- 不同随机种子下最终性能的方差。


## 2. 比较不同策略参数化

教程中的策略可以使用线性模型，也可以使用小型神经网络。比较不同模型容量时，应固定优化预算，并讨论：模型更强是否一定更容易优化？更大的参数空间是否会增加黑盒优化难度？

建议至少比较两种策略表示，例如线性策略与一层隐藏层的神经网络策略。


## 3. 比较优化设置

以 CMA-ES 或教程中的 `(1+1)-Active-CMA-ES` 为基础，研究下列因素之一或多个：

1. 初始步长；
2. 初始策略参数；
3. 每个候选策略使用多少个环境随机种子评价；
4. 使用平均回报、最差回报或二者组合做目标；
5. 固定环境交互预算下，不同策略维度对收敛速度的影响。


## 4. 公平评价

比较不同方法时，需要统一环境交互预算。一个候选策略如果用 $K$ 个随机种子评估，就消耗了大约 $K$ 倍的 rollout 成本，因此不能只比较优化迭代次数。

建议横轴使用累计环境 step 数或累计 rollout 数，并使用多个独立随机种子重复整个优化过程。


## 5. 推荐实验流程

1. 选择环境，例如 `CartPole-v1` 或教程中使用的另一个 Gymnasium 环境；
2. 定义策略模型与参数向量；
3. 定义 `evaluate_policy(theta, seeds)`；
4. 用进化策略搜索参数；
5. 记录 best-so-far、平均回报与交互预算；
6. 使用未参与优化的测试随机种子重新评价最终策略；
7. 对多个优化随机种子重复实验并绘制均值/方差。


In [ ]:
import numpy as np

def evaluate_policy(policy_from_theta, theta, rollout, env_name, seeds):
    """在多个环境随机种子上评价一个策略参数向量。"""
    returns = []
    policy = policy_from_theta(theta)
    for seed in seeds:
        history, _ = rollout(env_name, policy=policy, render=False, seed=seed)
        returns.append(sum(step[3] for step in history))
    return np.mean(returns), np.std(returns)


# 实验报告要求

报告至少包含：

- **实验目的**：希望验证什么问题；
- **方法**：策略结构、优化算法和关键超参数；
- **评价方法**：环境、训练/测试随机种子、预算和指标；
- **结果**：学习曲线和最终性能；
- **讨论**：为什么出现这些结果，哪些结论具有统计上的稳定性；
- **结论**：用几句话回答实验目的中提出的问题。

请避免只展示一条最好的学习曲线。策略优化具有明显随机性，多次独立实验是形成可信结论的必要条件。
